# Data Preparation

Der erste Schritt normalisiert die ROhdaten und reichert sie mit query-relevanten Daten an. Das Anreichern wird mit einem LLM durchgeführt. Die Daten prüfe ich nur stichprobenartig, sie gelten erstmal als so wahr.

- Rohdaten: Aus einem Vibecoding-Projekt.
- LLM zur Aufbereitung: Mistral
- Evaluation: Ein bisschen Pandas 
- Input: products_raw.json
- Output: products_enriched.json

Wegen mehrmaliger Crashes werden Beschreibungen und technische Daten erweitern jeweils die Rohdaten erweitern und ihre Ergebnisse zwischenspeichern, ehe sie zuletzt zusammegeführt werden. Es werden nur Rohdaten verwendet, die ausreichend Beschreibung UND technische Daten liefert. Somit wird auch ein JSONL File geschrieben um das Handling zu vereinfachen.

In [ ]:
import os
import json
import pandas as pd
from tqdm import tqdm
from mistralai import Mistral
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv('MISTRAL_API_KEY')
model = 'mistral-medium-2508'
client = Mistral(api_key=api_key, timeout_ms=500000)

def agent_request(system_prompt, schema, content):
            
    response = client.chat.complete(
        model = model,
        messages = [
            {
                'role': 'system',
                'content': system_prompt
            },
            {
                'role': 'user',
                'content': content,
            }
        ],
        response_format = {
            'type': 'json_object',
            'json_schema': schema,
            'strict': True
        }
    )

    return response

with open('../data/raw/products_raw.json', 'r') as f:
     products_raw = json.load(f)

products_filtered = [
    p for p in products_raw
    if len(p.get('description', '')) >= 100 and len(p.get('specs', [])) != 0
]

# Testing und Debuging
# products_filtered = products_filtered[:10]

# Fehlende ausgeben
raw_ids = set(item['id'] for item in products_raw)
filtered_ids = set(item['id'] for item in products_filtered)

print(f"Rohdaten:            {len(raw_ids)}")
print(f"Gefiltert:           {len(filtered_ids)}")
print()
print(raw_ids.difference(filtered_ids))

## Dataprep



In [ ]:
# Bereits aufbereitete Produkte auslesen
products_enriched = set()

if os.path.exists('../data/processed/products_enriched.jsonl'):
    with open('../data/processed/products_enriched.jsonl', 'r', encoding='utf-8') as f:
        for line in f:
            product = json.loads(line)
            products_enriched.add(product['id'])

products_to_process = [
    p for p in products_filtered
    if p['id'] not in products_enriched
]


print(f"Bereits verarbeitet: {len(products_enriched)}")
print(f"Noch zu verarbeiten: {len(products_to_process)}")

## Aufbereitung

In [ ]:
with open('../data/prompts/descs_agent.md', 'r') as f:
    descs_prompt = f.read()

with open('../data/prompts/descs_schema.json', 'r')as f:
    descs_schema = json.load(f)

with open('../data/prompts/specs_agent.md', 'r') as f:
    specs_prompt = f.read()

with open('../data/prompts/specs_schema.json', 'r') as f:
    specs_schema = json.load(f)

In [ ]:
# Produkte aufbereiten.
for product in tqdm(products_to_process, total=len(products_to_process)):

    # Specs brauch den Produktbezeichner
    specs_request = {
        'title': product['title'],
        'specs': product['specs']
    }

    #Requests
    desc_response = agent_request(descs_prompt, descs_schema, product['description'])
    specs_response = agent_request(specs_prompt, specs_schema, json.dumps(specs_request))

    # Zusammenbau neue Produktliste
    desc_json = json.loads(desc_response.choices[0].message.content)

    product_enriched = {
        'id': product['id'],
        'title': product['title'],
        'delivery_info': product['delivery_info'],
        'category': desc_json['category'],
        'descriptions': desc_json['descriptions'],
        'specs': json.loads(specs_response.choices[0].message.content),
        'usage': {
            'description': desc_response.usage.model_dump(),
            'specs': specs_response.usage.model_dump() 
        }
    }

    with open('../data/processed/products_enriched.jsonl', 'a', encoding='utf-8') as f:
        f.write(json.dumps(product_enriched, ensure_ascii=False) + '\n')

## Evaluation

Einmal nachsehen wie lange die Documents geworden sind und ob alle gefüllt wurden

In [ ]:
with open('../data/processed/products_enriched.jsonl', 'r', encoding='utf-8') as f:
    evaldata = [json.loads(line) for line in f]

In [ ]:
# Mengenvergleich
print(f"Rohdaten:  {len(raw_ids)}")
print()
print(f"Gefiltert: {len(filtered_ids)}")
print(f"Enriched:  {len(evaldata)}")


In [ ]:


specs_chunks = []
descs_chunks = []
costs = []

for product in evaldata:

    costs.append({
        'id': product['id'],
        'type': 'descs',
        'prompt_tokens': product['usage']['description']['prompt_tokens'],
        'completion_tokens': product['usage']['description']['completion_tokens'],
        'total_tokens': product['usage']['description']['total_tokens'],
    })

    costs.append({
        'id': product['id'],
        'type': 'specs',
        'prompt_tokens': product['usage']['specs']['prompt_tokens'],
        'completion_tokens': product['usage']['specs']['completion_tokens'],
        'total_tokens': product['usage']['specs']['total_tokens'],
    })

    for i, doc in enumerate(product.get('descriptions')):

        descs_chunks.append({
            'id': product['id'],
            'num': f"{i:02d}",
            'doc': doc,
            'len': len(doc),
            'words': len(doc.split())
        })

    for i, spec in enumerate(product.get('specs')):

        specs_chunks.append({
            'id': product['id'],
            'num': f"{i:02d}",
            'doc': spec['natural_language_description'],
            'len': len(spec['natural_language_description']),
            'words': len(spec['natural_language_description'].split())
        })

descs_df = pd.DataFrame(descs_chunks)
specs_df = pd.DataFrame(specs_chunks)
costs_df = pd.DataFrame(costs)

print(specs_df.info())
print(specs_df.head(5))
print(specs_df.columns)

In [ ]:
# Prüfen wieviele Absätze pro Produkt (min, max, mean)
print(f"Beschreibungen:\n{descs_df['words'].describe()}")
print()

# Prüfen wieviele Specs pro Produkt (min, max, mean)
print(f"Technische Daten:\n{specs_df['words'].describe()}")
print()

# Kurze und Lange Chunks anschauen
print(descs_df[descs_df['words'] <= 50].to_string())
print(specs_df[specs_df['words'] <= 5].to_string())